# NSC Synthetic Dataset Generator (2021–2025)

**Purpose:** Generate a synthetic dataset of South African National Senior Certificate (NSC) Grade 12 candidates for each year from 2021 to 2025, calibrated against the DBE NSC Examination Reports. This dataset is intended as a placeholder for the MIT 807 research project on first-year Computer Science student risk prediction at the University of Pretoria, while ethical clearance for the real UP dataset is pending.

**Author:** Thabo Chesane (u20507102)  
**Research topic:** Predictive model to identify first-year Computer Science students at risk

## What this notebook produces

Five CSV files `NSC_Synthetic_2021.csv` ... `NSC_Synthetic_2025.csv`  each containing roughly the same number of candidates as wrote the real NSC in that year:

| Year | Target candidates (full-time, from DBE reports) |
|------|----------------------------------|
| 2021 | 704 021 |
| 2022 | 725 146 |
| 2023 | 691 160 |
| 2024 | 705 291 |
| 2025 | 746 110 |



## Columns in each CSV

| Column | Description |
|--------|-------------|
| `candidate_id` | Unique 12-character hex ID |
| `year` | NSC year (2021–2025) |
| `province` | One of the 9 South African provinces, weighted by real enrolment share |
| `quintile` | 1 (poorest) – 5 (wealthiest) or 9 (Independent), weighted by real enrolment share |
| `fee_status` | `No Fee` (Q1–3) / `Fee Paying` (Q4–5) / `Independent` (Q9) |
| `candidate_type` | `Full-time` or `Part-time` |
| `progressed_learner` | Boolean — learner progressed from Gr 11 without meeting all requirements |
| `age` | Age at time of writing (most are 17–19) |
| `home_language` | Home language subject |
| `first_additional_language` | FAL subject |
| `has_mathematics` | True if candidate took Mathematics (vs Mathematical Literacy) |
| `mark_*` | 7 subject marks (0–100): Home Lang, FAL, LO, Maths/Lit, and 3 electives |
| `subject_*` | The subject taken in each slot |
| `num_subjects_taken` | Total count of subjects written (7 core + optional AP subjects) |
| `has_ap_mathematics` | True if took Advanced Programme Mathematics (IEB, rare) |
| `ap_mathematics_mark` | Mark if applicable, else NaN |
| `has_ap_english` | True if took AP English |
| `ap_english_mark` | Mark if applicable, else NaN |
| `overall_average` | Mean of the 7 core subject marks |
| `aps_score` | Admission Point Score calculated per University of Pretoria rules: top 6 20-credit subjects (LO excluded) on the NSC 1–7 scale. Max = 42. |
| `qualification_achieved` | `Bachelor` / `Diploma` / `Higher Certificate` / `NSC` / `Failed` — derived from marks per DBE rules |
| `bachelor_eligible` | True if qualification is Bachelor |
| `passed_nsc` | True if qualification is not `Failed` |

## Calibration approach

1. **Provincial enrolment shares** taken directly from Table 6.1.1 of each year's report.
2. **Quintile enrolment shares** taken from Table 6.1.11 (candidates by quintile).
3. **Subject pass rates** (% achieving 30%+) taken from Table 6.3.3 (non-language subjects).
4. **Quintile effect on marks** modeled as an additive shift on the underlying latent ability — calibrated so that the distribution of qualification types per quintile matches Table 6.1.10 / 6.1.11 of each report (e.g., Q5 Bachelor ~55–60%, Q1 Bachelor ~33–44% depending on year).
5. **Subject marks** are drawn from a truncated-normal latent ability per candidate, then shifted by the subject-specific difficulty to produce a realistic joint distribution (a strong candidate is strong across most subjects — essential for a risk-prediction model to learn from).
6. **NSC qualification thresholds** follow the official DBE rules: home language ≥40%, two other subjects ≥40%, three more ≥30%, LO ≥30%, and (for Bachelor) ≥50% in at least four 20-credit designated subjects.

## 1. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import secrets
from pathlib import Path
from scipy import stats


RNG_SEED = 20250807
rng = np.random.default_rng(RNG_SEED)


OUTPUT_DIR = Path("./synthetic_nsc_data")
OUTPUT_DIR.mkdir(exist_ok=True)


TARGET_COUNTS = {
    2021: 704_021,
    2022: 725_146,
    2023: 691_160,   
    2024: 705_291,
    2025: 746_110,
}



print("Configured for years:", list(TARGET_COUNTS.keys()))
print("Total synthetic candidates across all years:", f"{sum(TARGET_COUNTS.values()):,}")

## Calibration data extracted from NSC Examination Reports

All figures below come directly from the published DBE NSC Examination Reports for 2021–2025. The exact tables used are cited inline.

In [ ]:


PROVINCE_STATS = {
    2021: {
        "Eastern Cape":  (91_500, 73.0),
        "Free State":    (35_055, 85.7),
        "Gauteng":       (127_523, 82.8),
        "KwaZulu-Natal": (166_570, 76.8),
        "Limpopo":       (105_101, 66.7),
        "Mpumalanga":    (66_756, 73.6),
        "North West":    (41_081, 78.2),
        "Northern Cape": (12_726, 71.4),
        "Western Cape":  (57_709, 81.2),
    },
    2022: {
        "Eastern Cape":  (94_993, 77.3),
        "Free State":    (36_607, 88.5),
        "Gauteng":       (133_841, 84.4),
        "KwaZulu-Natal": (164_308, 83.0),
        "Limpopo":       (110_295, 72.1),
        "Mpumalanga":    (67_367, 76.8),
        "North West":    (43_823, 79.8),
        "Northern Cape": (13_574, 74.2),
        "Western Cape":  (60_338, 81.4),
    },
    2023: {
        "Eastern Cape":  (95_697, 81.42),
        "Free State":    (34_451, 89.03),
        "Gauteng":       (127_697, 85.38),
        "KwaZulu-Natal": (157_911, 86.36),
        "Limpopo":       (93_533, 79.54),
        "Mpumalanga":    (65_534, 76.95),
        "North West":    (41_418, 81.63),
        "Northern Cape": (12_842, 75.84),
        "Western Cape":  (62_077, 81.54),
    },
    2024: {
        "Eastern Cape":  (99_739, 84.98),
        "Free State":    (36_312, 90.99),
        "Gauteng":       (133_228, 88.41),
        "KwaZulu-Natal": (161_962, 89.52),
        "Limpopo":       (93_474, 85.01),
        "Mpumalanga":    (64_201, 84.99),
        "North West":    (40_575, 87.52),
        "Northern Cape": (12_937, 84.19),
        "Western Cape":  (62_863, 86.56),
    },
    2025: {
        "Eastern Cape":  (106_561, 84.17),
        "Free State":    (38_205, 89.33),
        "Gauteng":       (140_927, 89.06),
        "KwaZulu-Natal": (171_368, 90.60),
        "Limpopo":       (100_973, 86.15),
        "Mpumalanga":    (67_114, 86.55),
        "North West":    (40_913, 88.49),
        "Northern Cape": (14_084, 87.79),
        "Western Cape":  (65_965, 88.20),
    },
}



QUALIFICATION_RATES = {
    2021: {"Bachelor": 36.4, "Diploma": 25.2, "Higher Certificate": 14.8, "NSC": 0.0},
    2022: {"Bachelor": 38.4, "Diploma": 26.7, "Higher Certificate": 14.9, "NSC": 0.0},
    2023: {"Bachelor": 40.9, "Diploma": 27.2, "Higher Certificate": 14.8, "NSC": 0.0},
    2024: {"Bachelor": 47.8, "Diploma": 26.8, "Higher Certificate": 12.7, "NSC": 0.0},
    2025: {"Bachelor": 47.8, "Diploma": 26.8, "Higher Certificate": 12.7, "NSC": 0.0},
}



QUINTILE_ENROLMENT = {
    2021: {1: 156_716, 2: 159_054, 3: 174_743, 4: 83_405, 5: 106_201, 9: 23_902},
    2022: {1: 160_232, 2: 163_295, 3: 178_231, 4: 89_215, 5: 108_819, 9: 25_354},
    2023: {1: 151_786, 2: 148_996, 3: 167_905, 4: 86_926, 5: 107_121, 9: 28_426},
    2024: {1: 157_144, 2: 151_400, 3: 172_867, 4: 86_633, 5: 108_770, 9: 27_327},
    2025: {1: 168_202, 2: 161_287, 3: 184_849, 4: 90_361, 5: 111_053, 9: 29_208},
}



QUINTILE_BACHELOR_RATE = {
    2021: {1: 28.6, 2: 30.7, 3: 32.1, 4: 38.2, 5: 57.2, 9: 56.6},
    2022: {1: 33.0, 2: 33.6, 3: 34.9, 4: 37.4, 5: 55.6, 9: 59.2},
    2023: {1: 36.4, 2: 37.1, 3: 38.3, 4: 41.5, 5: 58.1, 9: 61.3},
    2024: {1: 44.4, 2: 45.0, 3: 44.3, 4: 45.6, 5: 60.3, 9: 62.2},
    2025: {1: 41.2, 2: 42.7, 3: 43.1, 4: 45.8, 5: 60.8, 9: 63.2},
}



SUBJECT_PASS_RATES = {
    2021: {"Accounting": 74.7, "Agricultural Sciences": 75.4, "Business Studies": 80.5,
           "Economics": 67.9, "Geography": 74.3, "History": 89.5, "Life Sciences": 71.5,
           "Mathematical Literacy": 74.5, "Mathematics": 57.6, "Physical Sciences": 69.0,
           "Consumer Studies": 98.0, "Tourism": 98.2, "CAT": 92.6, "IT": 93.0,
           "EGD": 92.8, "Life Orientation": 99.8, "Dramatic Arts": 99.1,
           "Religion Studies": 80.2, "Technical Mathematics": 60.1, "Technical Sciences": 87.1,
           "Hospitality Studies": 98.2, "Visual Arts": 98.4, "Music": 99.0},
    2022: {"Accounting": 75.4, "Agricultural Sciences": 75.8, "Business Studies": 76.7,
           "Economics": 71.5, "Geography": 81.3, "History": 88.2, "Life Sciences": 71.5,
           "Mathematical Literacy": 85.7, "Mathematics": 55.0, "Physical Sciences": 74.6,
           "Consumer Studies": 97.8, "Tourism": 98.0, "CAT": 92.6, "IT": 93.1,
           "EGD": 92.6, "Life Orientation": 99.8, "Dramatic Arts": 99.0,
           "Religion Studies": 80.5, "Technical Mathematics": 59.8, "Technical Sciences": 86.5,
           "Hospitality Studies": 98.1, "Visual Arts": 98.4, "Music": 98.9},
    2023: {"Accounting": 76.8, "Agricultural Sciences": 80.5, "Business Studies": 81.8,
           "Economics": 74.5, "Geography": 86.2, "History": 87.7, "Life Sciences": 75.6,
           "Mathematical Literacy": 82.3, "Mathematics": 63.5, "Physical Sciences": 76.2,
           "Consumer Studies": 98.0, "Tourism": 98.3, "CAT": 93.1, "IT": 93.5,
           "EGD": 93.0, "Life Orientation": 99.8, "Dramatic Arts": 99.1,
           "Religion Studies": 81.0, "Technical Mathematics": 62.5, "Technical Sciences": 87.0,
           "Hospitality Studies": 98.2, "Visual Arts": 98.5, "Music": 99.0},
    2024: {"Accounting": 81.2, "Agricultural Sciences": 86.9, "Business Studies": 86.0,
           "Economics": 80.5, "Geography": 89.5, "History": 90.3, "Life Sciences": 80.8,
           "Mathematical Literacy": 86.1, "Mathematics": 69.1, "Physical Sciences": 75.6,
           "Consumer Studies": 98.1, "Tourism": 98.5, "CAT": 93.5, "IT": 93.8,
           "EGD": 93.2, "Life Orientation": 99.8, "Dramatic Arts": 99.2,
           "Religion Studies": 81.2, "Technical Mathematics": 64.0, "Technical Sciences": 87.5,
           "Hospitality Studies": 98.2, "Visual Arts": 98.6, "Music": 99.1},
    2025: {"Accounting": 77.9, "Agricultural Sciences": 81.2, "Business Studies": 88.4,
           "Economics": 80.8, "Geography": 91.1, "History": 91.8, "Life Sciences": 83.4,
           "Mathematical Literacy": 86.2, "Mathematics": 64.0, "Physical Sciences": 77.3,
           "Consumer Studies": 98.1, "Tourism": 98.5, "CAT": 93.5, "IT": 93.8,
           "EGD": 93.2, "Life Orientation": 99.8, "Dramatic Arts": 99.2,
           "Religion Studies": 81.5, "Technical Mathematics": 65.0, "Technical Sciences": 87.8,
           "Hospitality Studies": 98.3, "Visual Arts": 98.7, "Music": 99.2},
}



FT_PT_SPLIT = {
    2021: (733_540, 163_950),
    2022: (752_003, 168_631),
    2023: (715_719, 182_056),
    2024: (724_156, 155_948),
    2025: (764_014, 137_776),
}

print("Calibration data loaded for", len(PROVINCE_STATS), "years.")
print("Subjects tracked per year:", len(SUBJECT_PASS_RATES[2021]))

## Subject structure and difficulty calibration

Every NSC candidate takes **7 subjects**:

1. A **Home Language** (Zulu, Xhosa, Afrikaans, English, Sesotho, etc.)
2. A **First Additional Language** (usually English)
3. **Life Orientation** (compulsory, nearly universal pass — 99.8%)
4. Either **Mathematics** OR **Mathematical Literacy**
5. Three further elective subjects

We also add optional **Advanced Programme (AP) subjects** — these are IEB-set subjects written by a small minority of (mostly Q5) candidates who want to push into university with extra credentials.

In [ ]:


HOME_LANGUAGES = {
    "isiZulu HL":        0.235,
    "isiXhosa HL":       0.175,
    "English HL":        0.150,
    "Afrikaans HL":      0.100,
    "Sepedi HL":         0.095,
    "Setswana HL":       0.075,
    "Sesotho HL":        0.065,
    "Xitsonga HL":       0.045,
    "siSwati HL":        0.025,
    "Tshivenda HL":      0.025,
    "isiNdebele HL":     0.010,
}


FAL_FOR_ENGLISH_HL = {"Afrikaans FAL": 0.85, "isiZulu FAL": 0.10, "isiXhosa FAL": 0.05}
FAL_FOR_OTHER_HL  = {"English FAL": 1.0}



ELECTIVE_SUBJECTS = {
    "Life Sciences":          0.48,  
    "Geography":              0.45,  
    "History":                0.30, 
    "Business Studies":       0.30,  
    "Physical Sciences":      0.26,  
    "Tourism":                0.26,  
    "Agricultural Sciences":  0.15,  
    "Economics":              0.16,  
    "Accounting":             0.13,  
    "Consumer Studies":       0.06,  
    "CAT":                    0.05,  
    "EGD":                    0.05,  
    "Religion Studies":       0.02,
    "Dramatic Arts":          0.02,
    "Technical Mathematics":  0.02,
    "Technical Sciences":     0.02,
    "Hospitality Studies":    0.01,
    "Visual Arts":            0.01,
    "Music":                  0.005,
    "IT":                     0.005,
}


MATHS_UPTAKE_BY_QUINTILE = {1: 0.27, 2: 0.29, 3: 0.33, 4: 0.45, 5: 0.62, 9: 0.72}



QUINTILE_ABILITY_SHIFT = {
    1: -0.20,
    2: -0.15,
    3: -0.10,
    4:  0.15,
    5:  0.55
}



PROVINCE_ABILITY_SHIFT = {
    "Free State":     0.12,
    "Gauteng":        0.15,
    "Western Cape":   0.12,
    "KwaZulu-Natal":  0.02,
    "North West":     0.00,
    "Mpumalanga":    -0.05,
    "Northern Cape": -0.05,
    "Eastern Cape":  -0.10,
    "Limpopo":       -0.15,
}

print("Subject and quintile calibration loaded.")

## Core generator functions

The approach: each candidate has a **latent ability $\theta \sim \mathcal{N}(0, 1)$** shifted by their quintile and province. Each subject has a **difficulty parameter $\beta_s$** calibrated so the marginal % achieving ≥30% matches the DBE report. A candidate's mark in subject $s$ is then drawn from a normal distribution centred on $\theta - \beta_s$, with within-candidate noise. This reproduces the joint structure — a strong candidate is strong across most subjects — which is critical when downstream models will be trained on subject marks as features.

In [ ]:
def hex_id() -> str:
    """Generate a unique 12-character hex ID."""
    return secrets.token_hex(6).upper()


def solve_subject_difficulty(pass_rate_pct: float, subject_sd: float = 18.0, threshold: float = 30.0) -> float:
    """
    Given a target pass rate (% of candidates achieving >= threshold) under a
    national-average ability (theta = 0), solve for the mark-distribution
    mean such that P(mark >= threshold) == pass_rate_pct / 100.

    We model each subject mark as:
         mark ~ Normal(mu_s + scale*(theta + shift), subject_sd)
    with mu_s chosen here so that, at theta = 0 and shift = 0,
    the marginal pass rate is as reported.

    Note: DBE reports the Home Language pass rate at the >=40% threshold
    (since a 40% HL mark is required for NSC), whereas non-language subjects
    are reported at the >=30% threshold. Use the `threshold` argument accordingly.
    """
    p = max(min(pass_rate_pct / 100.0, 0.999), 0.001)
   
    mu = threshold + subject_sd * stats.norm.ppf(p)
    return float(mu)


def draw_subject_mark(
    theta: np.ndarray,
    subject_mean: float,
    ability_weight: float = 10.0,
    noise_sd: float = 12.0,
) -> np.ndarray:
    """
    Draw a subject mark vector for an array of candidate abilities theta.

    Parameters
    
    theta : ndarray
        Latent ability of each candidate (already incl. quintile & province shifts).
    subject_mean : float
        Mean mark for a candidate of ability theta = 0 in this subject.
    ability_weight : float
        How much a 1-SD increase in ability lifts the mark (in mark points).
    noise_sd : float
        Within-candidate, per-subject noise. Captures variation across subjects
        for the same candidate.
    """
    marks = subject_mean + ability_weight * theta + rng.normal(0, noise_sd, size=theta.shape)
    return np.clip(marks, 0, 100)


def classify_qualification(
    home_lang: float,
    fal: float,
    lo: float,
    maths_or_ml: float,
    e1: float, e2: float, e3: float,
    took_maths: bool,
) -> str:
    """
    Apply simplified NSC qualification rules per DBE:

      NSC pass requirements:
        - Home Language >= 40%
        - Life Orientation >= 30%
        - 2 other subjects >= 40%
        - 3 other subjects >= 30%  (amended rule allows 1 subject below 30%)

      Qualification types:
        - Higher Certificate: basic NSC pass
        - Diploma: 3+ subjects at 50%+, 5+ subjects at 40%+
        - Bachelor: 5+ subjects at 50%+, 3+ subjects at 60%+
          (approximates the 4-designated-subject-at-50% rule)

    Calibrated against DBE Tables 6.2.2 / 6.1.3 / 6.1.4 across 2021-2025.
    """
    subjects = [home_lang, fal, lo, maths_or_ml, e1, e2, e3]
    # 5 subjects excluding HL and LO from which the 2@40 + 3@30 requirement is counted
    nonhome_nonLO = [fal, maths_or_ml, e1, e2, e3]
    count_40 = sum(1 for m in nonhome_nonLO if m >= 40)
    count_30 = sum(1 for m in nonhome_nonLO if m >= 30)

    # Basic NSC pass
    if home_lang < 40 or lo < 30 or count_40 < 2:
        return "Failed"
    # Amended NSC allows 1 of the 5 non-HL/LO subjects to fall below 30%
    if count_30 < 4:
        return "Failed"

    # Grade into Bachelor / Diploma / Higher Certificate
    count_subjects_50 = sum(1 for m in subjects if m >= 50)
    count_subjects_60 = sum(1 for m in subjects if m >= 60)
    count_subjects_40 = sum(1 for m in subjects if m >= 40)

    # Bachelor: strong all-round performance
    if count_subjects_50 >= 5 and count_subjects_60 >= 3 and home_lang >= 40:
        return "Bachelor"
    # Diploma: solid performance in most subjects
    if count_subjects_50 >= 3 and count_subjects_40 >= 5:
        return "Diploma"
    return "Higher Certificate"


print("Generator utility functions defined.")

In [ ]:
def generate_year(year: int, n_candidates: int) -> pd.DataFrame:
    """
    Generate the full synthetic candidate dataset for one NSC year.
    """
    prov_stats = PROVINCE_STATS[year]
    quint_enrol = QUINTILE_ENROLMENT[year]
    subject_rates = SUBJECT_PASS_RATES[year]

    
    provinces = list(prov_stats.keys())
    prov_weights = np.array([prov_stats[p][0] for p in provinces], dtype=float)
    prov_weights /= prov_weights.sum()
    prov_assignment = rng.choice(provinces, size=n_candidates, p=prov_weights)

    
    quintiles = list(quint_enrol.keys())
    q_weights = np.array([quint_enrol[q] for q in quintiles], dtype=float)
    q_weights /= q_weights.sum()
    quintile_assignment = rng.choice(quintiles, size=n_candidates, p=q_weights)

    
    def fee_status(q):
        if q in (1, 2, 3):
            return "No Fee"
        if q in (4, 5):
            return "Fee Paying"
        return "Independent"
    fee = np.array([fee_status(q) for q in quintile_assignment])

    # Full-time / Part-time (small part-time share among writers) 
    # Most PT writers are not pulled through the same subject pipeline, but a
    # portion of the writers pool are PT. We include ~5% PT in the generated
    # FT-writer-target set for realism in the downstream dataset.
    ft_count, pt_count = FT_PT_SPLIT[year]
    pt_share = pt_count / (ft_count + pt_count)
    # Apply a dampening — only some PTs are included in the writer pool
    pt_share_effective = pt_share * 0.30
    candidate_type = np.where(
        rng.random(n_candidates) < pt_share_effective,
        "Part-time",
        "Full-time",
    )

    # Progressed learner flag (~8% of FT candidates)
    # DBE 2024 report: ~56 536 progressed learners out of ~724k FT
    progressed = (rng.random(n_candidates) < 0.08) & (candidate_type == "Full-time")

    # Demographics
    # Age distribution: mostly 17-19, with tail for progressed and part-time
    age = np.round(rng.normal(18.3, 0.9, size=n_candidates)).astype(int)
    age = np.clip(age, 15, 45)
    # Part-time and progressed tend to be older
    age = np.where(candidate_type == "Part-time", age + rng.integers(2, 8, n_candidates), age)
    age = np.where(progressed, age + 1, age)
    age = np.clip(age, 15, 55)

    # Latent ability 
    # theta ~ N(0,1) with shifts for quintile, province, progressed status
    base_theta = rng.normal(0, 1, size=n_candidates)
    q_shift = np.array([QUINTILE_ABILITY_SHIFT[q] for q in quintile_assignment])
    p_shift = np.array([PROVINCE_ABILITY_SHIFT[p] for p in prov_assignment])
    prog_shift = np.where(progressed, -0.35, 0.0)
    pt_shift = np.where(candidate_type == "Part-time", -0.40, 0.0)
    theta = base_theta + q_shift + p_shift + prog_shift + pt_shift

    # Language assignment 
    hl_choices = list(HOME_LANGUAGES.keys())
    hl_weights = np.array(list(HOME_LANGUAGES.values()))
    hl_weights = hl_weights / hl_weights.sum()
    home_language = rng.choice(hl_choices, size=n_candidates, p=hl_weights)

    # FAL follows from HL
    fal_list = []
    for hl in home_language:
        if hl == "English HL":
            choices = list(FAL_FOR_ENGLISH_HL.keys())
            weights = list(FAL_FOR_ENGLISH_HL.values())
        else:
            choices = list(FAL_FOR_OTHER_HL.keys())
            weights = list(FAL_FOR_OTHER_HL.values())
        fal_list.append(rng.choice(choices, p=np.array(weights) / np.sum(weights)))
    fal_language = np.array(fal_list)

    
    # Probability of taking Mathematics depends on quintile and ability
    base_maths_prob = np.array([MATHS_UPTAKE_BY_QUINTILE[q] for q in quintile_assignment])
    # Higher ability -> more likely to take Maths
    ability_adj = stats.norm.cdf(theta * 0.5)
    maths_prob = np.clip(0.65 * base_maths_prob + 0.35 * ability_adj, 0.0, 0.95)
    has_mathematics = rng.random(n_candidates) < maths_prob

    
    elective_names = list(ELECTIVE_SUBJECTS.keys())
    elective_weights = np.array(list(ELECTIVE_SUBJECTS.values()))
    elective_weights = elective_weights / elective_weights.sum()

    # Vectorised draw — for each candidate sample 3 distinct electives
    # We do this in batches for speed
    elective1 = np.empty(n_candidates, dtype=object)
    elective2 = np.empty(n_candidates, dtype=object)
    elective3 = np.empty(n_candidates, dtype=object)
    # Draw 3 with replacement then resolve duplicates (fast and close enough for this scale)
    for i in range(n_candidates):
        picks = rng.choice(elective_names, size=3, replace=False, p=elective_weights)
        elective1[i], elective2[i], elective3[i] = picks[0], picks[1], picks[2]

   
    # Language subjects:
    # - Home Language: DBE reports >=40% pass rate at ~91-99%; calibrate against 40%
    # - FAL: ~91% pass at 30%
    # - LO: 99.8% pass at 30% (practically universal)
    home_lang_mean = solve_subject_difficulty(95.0, threshold=40.0)
    fal_mean       = solve_subject_difficulty(93.0, threshold=30.0)
    lo_mean        = solve_subject_difficulty(99.8, threshold=30.0)

    # Maths / Mathematical Literacy
    maths_mean = solve_subject_difficulty(subject_rates["Mathematics"])
    ml_mean    = solve_subject_difficulty(subject_rates["Mathematical Literacy"])

    home_lang_mark = draw_subject_mark(theta, home_lang_mean, ability_weight=11, noise_sd=11)
    fal_mark       = draw_subject_mark(theta, fal_mean, ability_weight=10, noise_sd=12)
    lo_mark        = draw_subject_mark(theta, lo_mean, ability_weight=6,  noise_sd=8)
    # Maths / ML slot
    maths_ml_subject = np.where(has_mathematics, "Mathematics", "Mathematical Literacy")
    maths_ml_mark    = np.where(
        has_mathematics,
        draw_subject_mark(theta, maths_mean, ability_weight=14, noise_sd=13),
        draw_subject_mark(theta, ml_mean,    ability_weight=9,  noise_sd=11),
    )

    # Elective marks — each elective has its own subject-specific mean
    subject_means = {s: solve_subject_difficulty(subject_rates[s]) for s in elective_names}

    def elective_draw(subjects_arr):
        means = np.array([subject_means[s] for s in subjects_arr])
        marks = means + 10 * theta + rng.normal(0, 12, size=n_candidates)
        return np.clip(marks, 0, 100)

    e1_mark = elective_draw(elective1)
    e2_mark = elective_draw(elective2)
    e3_mark = elective_draw(elective3)

    #  Advanced Programme (AP) subjects 
    # AP Maths/English are IEB extras — uptake is rare and concentrated in Q5/Q9
    # Roughly 4000-8000 AP Maths candidates nationally per year (0.5-1% of writers)
    ap_maths_prob = np.where(
        np.isin(quintile_assignment, [5, 9]) & has_mathematics & (theta > 0.5),
        0.08,   # 8% among strong Q5/Q9 Maths takers
        0.001,  # very rare otherwise
    )
    has_ap_mathematics = rng.random(n_candidates) < ap_maths_prob
    # AP Maths mean is lower than regular Maths because it's more difficult, 45% average
    ap_maths_mark = np.where(
        has_ap_mathematics,
        np.clip(45 + 11 * theta + rng.normal(0, 14, size=n_candidates), 0, 100),
        np.nan,
    )

    ap_english_prob = np.where(
        np.isin(quintile_assignment, [5, 9]) & (theta > 0.4),
        0.05,
        0.0005,
    )
    has_ap_english = rng.random(n_candidates) < ap_english_prob
    ap_english_mark = np.where(
        has_ap_english,
        np.clip(55 + 10 * theta + rng.normal(0, 12, size=n_candidates), 0, 100),
        np.nan,
    )

    # Qualification classification 
    quals = np.array([
        classify_qualification(hl, f, lo_, ml, m1, m2, m3, bool(hm))
        for hl, f, lo_, ml, m1, m2, m3, hm in zip(
            home_lang_mark, fal_mark, lo_mark, maths_ml_mark,
            e1_mark, e2_mark, e3_mark, has_mathematics,
        )
    ])

    # Overall average over the 7 core subjects 
    overall_avg = np.round(
        (home_lang_mark + fal_mark + lo_mark + maths_ml_mark + e1_mark + e2_mark + e3_mark) / 7.0,
        2,
    )

    #   University of Pretoria APS score 
    # UP rules:
    #   - Use the best 6 of the candidate's 20-credit subjects
    #   - Life Orientation is excluded (it is a 10-credit subject)
    #   - Each subject is converted to the NSC 1–7 scale of achievement:
    #       80-100% = 7, 70-79% = 6, 60-69% = 5, 50-59% = 4,
    #       40-49%  = 3, 30-39% = 2,  0-29%  = 1
    #   - AP subjects are IEB extras and are not standard 20-credit NSC
    #     subjects, so UP does not include them in the APS calculation itself.
    #     Some faculties award small bonus points for AP Mathematics passed
    #     at >=60%, but that is programme-specific and outside the standard APS.
    #   - Maximum APS = 42 (top 6 subjects all achieving level 7)
    def pct_to_nsc_level(marks: np.ndarray) -> np.ndarray:
        levels = np.zeros_like(marks, dtype=int)
        levels = np.where(marks >= 80, 7, levels)
        levels = np.where((marks >= 70) & (marks < 80), 6, levels)
        levels = np.where((marks >= 60) & (marks < 70), 5, levels)
        levels = np.where((marks >= 50) & (marks < 60), 4, levels)
        levels = np.where((marks >= 40) & (marks < 50), 3, levels)
        levels = np.where((marks >= 30) & (marks < 40), 2, levels)
        levels = np.where(marks < 30, 1, levels)
        return levels

    # Stack the six 20-credit subject marks (HL, FAL, Maths/ML, 3 electives) per candidate.
    # LO is deliberately NOT in this stack.
    six_subj_marks = np.stack([
        home_lang_mark, fal_mark, maths_ml_mark, e1_mark, e2_mark, e3_mark,
    ], axis=1)
    six_subj_levels = pct_to_nsc_level(six_subj_marks)
    # Sum of the top 6 (all 6, since we have exactly 6 eligible subjects)
    aps_score = six_subj_levels.sum(axis=1)

    #  Total number of subjects taken 
    # Every candidate takes the 7 core subjects (HL, FAL, LO, Maths/ML, 3 electives)
    # plus zero or more AP subjects (AP Maths, AP English).
    num_subjects_taken = 7 + has_ap_mathematics.astype(int) + has_ap_english.astype(int)

    #  Assemble DataFrame 
    df = pd.DataFrame({
        "candidate_id":          [hex_id() for _ in range(n_candidates)],
        "year":                  year,
        "province":              prov_assignment,
        "quintile":              quintile_assignment,
        "fee_status":            fee,
        "candidate_type":        candidate_type,
        "progressed_learner":    progressed,
        "age":                   age,
        "home_language":         home_language,
        "first_additional_lang": fal_language,
        "has_mathematics":       has_mathematics,
        "subject_home_language": home_language,
        "mark_home_language":    np.round(home_lang_mark, 1),
        "subject_fal":           fal_language,
        "mark_fal":              np.round(fal_mark, 1),
        "subject_life_orientation": "Life Orientation",
        "mark_life_orientation": np.round(lo_mark, 1),
        "subject_maths_or_ml":   maths_ml_subject,
        "mark_maths_or_ml":      np.round(maths_ml_mark, 1),
        "subject_elective_1":    elective1,
        "mark_elective_1":       np.round(e1_mark, 1),
        "subject_elective_2":    elective2,
        "mark_elective_2":       np.round(e2_mark, 1),
        "subject_elective_3":    elective3,
        "mark_elective_3":       np.round(e3_mark, 1),
        "has_ap_mathematics":    has_ap_mathematics,
        "ap_mathematics_mark":   np.round(ap_maths_mark, 1),
        "has_ap_english":        has_ap_english,
        "ap_english_mark":       np.round(ap_english_mark, 1),
        "num_subjects_taken":    num_subjects_taken,
        "overall_average":       overall_avg,
        "aps_score":             aps_score,
        "qualification_achieved": quals,
        "bachelor_eligible":     (quals == "Bachelor"),
        "passed_nsc":            (quals != "Failed"),
    })

    return df

print("Year generator defined.")

## Generate all five years and save CSVs

This cell runs the generator for 2021–2025 and saves each year to its own CSV file in `./synthetic_nsc_data/`.

In [ ]:
summary_rows = []

for year, n in TARGET_COUNTS.items():
    print(f"\nGenerating {year} — target {n:,} candidates ...")
    df = generate_year(year, n)
    out_path = OUTPUT_DIR / f"NSC_Synthetic_{year}.csv"
    df.to_csv(out_path, index=False)
    print(f"  Saved → {out_path}")

    # Collect headline stats for validation
    qual_counts = df["qualification_achieved"].value_counts(normalize=True) * 100
    summary_rows.append({
        "year": year,
        "candidates": len(df),
        "pct_pass":      round((df["qualification_achieved"] != "Failed").mean() * 100, 2),
        "pct_bachelor":  round(qual_counts.get("Bachelor", 0), 2),
        "pct_diploma":   round(qual_counts.get("Diploma", 0), 2),
        "pct_hc":        round(qual_counts.get("Higher Certificate", 0), 2),
        "pct_failed":    round(qual_counts.get("Failed", 0), 2),
        "pct_maths":     round(df["has_mathematics"].mean() * 100, 2),
        "maths_avg":     round(df.loc[df["has_mathematics"], "mark_maths_or_ml"].mean(), 2),
        "ml_avg":        round(df.loc[~df["has_mathematics"], "mark_maths_or_ml"].mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n=== Generation complete ===")
summary_df

## Validation compare synthetic vs. official NSC statistics

Compare the aggregate metrics from the generated data against the real NSC report figures to confirm calibration.

In [ ]:
validation_rows = []

for year in TARGET_COUNTS:
    df = pd.read_csv(OUTPUT_DIR / f"NSC_Synthetic_{year}.csv")
    # National pass rate
    reported_pass = sum(
        v[0] * v[1] / 100 for v in PROVINCE_STATS[year].values()
    ) / sum(v[0] for v in PROVINCE_STATS[year].values()) * 100
    synthetic_pass = (df["qualification_achieved"] != "Failed").mean() * 100

    validation_rows.append({
        "year":                  year,
        "reported_pass_rate":    round(reported_pass, 2),
        "synthetic_pass_rate":   round(synthetic_pass, 2),
        "reported_bachelor":     QUALIFICATION_RATES[year]["Bachelor"],
        "synthetic_bachelor":    round((df["qualification_achieved"] == "Bachelor").mean() * 100, 2),
        "reported_maths_passrate":   SUBJECT_PASS_RATES[year]["Mathematics"],
        "synthetic_maths_passrate": round(
            (df.loc[df["has_mathematics"], "mark_maths_or_ml"] >= 30).mean() * 100, 2,
        ),
        "reported_ml_passrate":      SUBJECT_PASS_RATES[year]["Mathematical Literacy"],
        "synthetic_ml_passrate":   round(
            (df.loc[~df["has_mathematics"], "mark_maths_or_ml"] >= 30).mean() * 100, 2,
        ),
    })

validation_df = pd.DataFrame(validation_rows)
print("Validation — synthetic aggregates vs. real NSC report figures\n")
validation_df

In [ ]:

import warnings
warnings.filterwarnings("ignore")

year = 2025
df = pd.read_csv(OUTPUT_DIR / f"NSC_Synthetic_{year}.csv")

by_quintile = (
    df.groupby("quintile").agg(
        candidates=("candidate_id", "count"),
        synthetic_bachelor_pct=("bachelor_eligible", lambda x: round(x.mean() * 100, 2)),
        synthetic_pass_pct=("passed_nsc", lambda x: round(x.mean() * 100, 2)),
        synthetic_maths_uptake=("has_mathematics", lambda x: round(x.mean() * 100, 2)),
    )
)
by_quintile["reported_bachelor_pct"] = by_quintile.index.map(QUINTILE_BACHELOR_RATE[year])
print(f"Quintile-level validation for {year}\n")
by_quintile

In [ ]:

by_province = (
    df.groupby("province").agg(
        candidates=("candidate_id", "count"),
        synthetic_pass_pct=("passed_nsc", lambda x: round(x.mean() * 100, 2)),
    )
)
by_province["reported_wrote"]    = by_province.index.map(lambda p: PROVINCE_STATS[year][p][0])
by_province["reported_pass_pct"] = by_province.index.map(lambda p: PROVINCE_STATS[year][p][1])
print(f"Provincial validation for {year}\n")
by_province

In [ ]:
sample = df.sample(10, random_state=42).reset_index(drop=True)
print("Sample of 10 synthetic candidates from 2025:")
sample

## Use the data in your model-building pipeline

You now have five calibrated CSVs in `./synthetic_nsc_data/`. For the MIT 807 predictive model of first-year CS risk, a natural next step is:

1. **Filter** the dataset to candidates plausibly enrolling in Computer Science at UP:  
   `has_mathematics == True`, `mark_maths_or_ml >= 50`, `bachelor_eligible == True`, and usually `mark_elective_1/2/3` including Physical Sciences or IT/CAT at 50%+.
2. **Synthesise a label** for first-year risk. The simplest option is to combine NSC performance with a hidden ability + noise model — candidates with lower Maths marks, non-English HL+FAL structure, and lower-quintile backgrounds should have higher first-year failure probability.
3. **Train baseline models** (logistic regression, Random Forest, Gradient Boosting) with proper class-imbalance handling and report precision/recall/F1 per your methodology section.
4. When ethical clearance arrives, swap the synthetic CSVs for the real UP data and re-run — your feature engineering code should transfer.

## Notes on limitations

- This is **synthetic** data intended for pipeline testing and methodology validation — no real candidates are represented.
- Joint distributions are modelled via a single latent ability — real candidates have multi-dimensional ability (verbal vs quantitative) that this collapses.
- District, school, and fine-grained language breakdowns use broad approximations. More granularity can be added by extending the tables in Section 2.
- The DBE reports do not publish full mark distributions (only pass-rate bands), so within-subject mark distributions here are inferred from the reported ≥30% / ≥40% / ≥50% bands.